# E08 · 00 从 Semantic Scholar 静态检索页提取 PDF 下载 URL

输入：`data/raw/e08/raw_html/*.html` —— 浏览器"另存为"下来的 Semantic Scholar
检索结果页（SERP）静态快照，每页 10 条结果。

输出：每条结果一行的表，含标题、元数据和**候选下载 URL**，供后续按需抓取 PDF。

**口径声明**

- 用关键词直接检索，筛选 computer science
- 这些 HTML 是**某一时刻的快照**，S2 的排序、徽章、开放获取状态都会随时间变化；
  文件名里的日期（若有）就是快照日期，没有则以文件 mtime 为准。
- `pdf_url` 是**页面上声明的链接，不是"已验证可下载的 PDF"**。ACM / IEEE 的
  链接常常需要订阅或会跳转到落地页；只有 `url_kind == "pdf"` 且域名是 arxiv /
  vldb / 作者主页一类的，才大概率能直接拿到文件。真正能不能下到，要到下载那一步
  才算数，本 notebook 不做这个判断。
- 本 notebook 是**探索面**，不是流水线。若这套解析要被文档引用的数字依赖，
  按 AGENTS.md 应先提升为 `experiments/e08/` 下的脚本，产物写进 `data/processed/e08/`。

In [ ]:
import json
import re
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlsplit

import pandas as pd
from bs4 import BeautifulSoup

REPO = Path.cwd()
while not (REPO / "AGENTS.md").exists():          # notebook 可能从任意目录启动
    REPO = REPO.parent

RAW_DIR = REPO / "data/raw/e08/raw_html"
OUT_DIR = Path.cwd() / "_scratch"                 # 探索产物，不进 data/processed

html_files = sorted(RAW_DIR.glob("*.html"))
for p in html_files:
    mtime = datetime.fromtimestamp(p.stat().st_mtime, timezone.utc).date()
    print(f"{p.name}  ({p.stat().st_size/1024:.0f} KiB, mtime {mtime})")

## 页面结构

一页 SERP 的每条结果是一个 `div.cl-paper-row[data-paper-id]`，里面和下载有关的锚点有三类：

| 位置 | 选择器 | 含义 |
| --- | --- | --- |
| 开放获取徽章 | `div.paper-badge-list a` 内含 `div[data-test-id^="paper-badge-"]` | S2 认为可公开获取的全文。实测两种：`paper-badge-unpaywall`（Unpaywall 给的，多为 `dl.acm.org/doi/pdf/...`）和 `paper-badge-open_access`（arXiv / VLDB / 作者主页的 `.pdf`） |
| 主链接 | `a[data-test-id="paper-link"]` | 出版方入口。属性 `data-heap-link-type`（acm/arxiv/ieee/doi/dblp/publisher）与 `data-heap-direct-pdf-link`（arXiv 条目为 `true`，直接就是 `.pdf`） |
| 阅读器 | `a.cl-paper-action__reader-link` | S2 自己托管的阅读器页，不是文件直链，留作兜底线索 |

三类都可能缺席：既无徽章也非 arXiv 的条目，只剩出版方落地页。

In [ ]:
BADGE_PREFIX = "paper-badge-"


def _text(node, sep=" "):
    return node.get_text(sep, strip=True) if node else None


# 粗分链接形态：pdf=文件直链 / doi=DOI 跳转 / landing=落地页
def _url_kind(url: str) -> str:
    path = urlsplit(url).path.lower()
    if path.endswith(".pdf") or "/doi/pdf/" in path or "/pdf/" in path:
        return "pdf"
    if urlsplit(url).netloc.endswith("doi.org"):
        return "doi"
    return "landing"


def parse_row(row, source: str) -> dict:
    paper_id = row.get("data-paper-id")
    title_a = row.select_one('a[data-test-id="title-link"]')

    # 候选链接：徽章 > 主链接 > 阅读器，保留来源以便排障
    candidates = []
    for a in row.select("div.paper-badge-list a[href]"):
        badge = a.find("div", attrs={"data-test-id": True})
        kind = (badge["data-test-id"] if badge else "badge").removeprefix(BADGE_PREFIX)
        candidates.append({"via": f"badge:{kind}", "url": a["href"]})
    for a in row.select('a[data-test-id="paper-link"][href]'):
        candidates.append({
            "via": f"paper-link:{a.get('data-heap-link-type')}",
            "url": a["href"],
            "direct_pdf_flag": a.get("data-heap-direct-pdf-link") == "true",
        })
    for c in candidates:
        c["url_kind"] = _url_kind(c["url"])
        c["host"] = urlsplit(c["url"]).netloc

    reader_a = row.select_one("a.cl-paper-action__reader-link[href]")
    cites = row.select_one('[data-test-id="total-citations-stat"][aria-label]')

    return {
        "source": source,
        "paper_id": paper_id,
        "title": _text(title_a.select_one("h2.cl-paper-title")) if title_a else None,
        "authors": [_text(a) for a in row.select("a.cl-paper-authors__author-box")],
        "venue": _text(row.select_one("span.cl-paper-venue")),
        "pub_date": _text(row.select_one("span.cl-paper-pubdates")),
        "citations": int(m.group(1)) if cites and (m := re.search(r"(\d+)", cites["aria-label"])) else None,
        "tldr": _text(row.select_one("div.tldr-abstract-replacement span span")),
        "s2_url": title_a["href"] if title_a else None,
        "reader_url": reader_a["href"] if reader_a else None,
        "candidates": candidates,
    }


def parse_page(path: Path) -> list[dict]:
    soup = BeautifulSoup(path.read_text(encoding="utf-8"), "html.parser")
    query = (soup.title.string or "").removesuffix(" | Semantic Scholar").strip()
    rows = soup.select("div.cl-paper-row")
    out = []
    for rank, row in enumerate(rows, 1):
        rec = parse_row(row, path.name)
        rec["query"] = query
        rec["rank"] = rank
        out.append(rec)
    return out


records = [rec for p in html_files for rec in parse_page(p)]
assert records, "没有解析到任何结果行——检查 S2 页面结构是否已变"
assert all(r["paper_id"] and r["title"] for r in records), "存在缺 paper_id/title 的行"
print(f"{len(html_files)} 个页面 -> {len(records)} 条结果，"
      f"{len({r['paper_id'] for r in records})} 个去重 paper_id")

## 选一个下载 URL

优先级按"拿到文件的把握"排：

1. 徽章里形态为 `pdf` 的直链（arXiv / VLDB / ACM `doi/pdf` / 作者主页）；
2. 主链接里 `data-heap-direct-pdf-link=true` 的（arXiv）；
3. 徽章里的 DOI 跳转 —— 需要跟随重定向才知道是不是 PDF；
4. 出版方落地页 —— 多半要订阅或手工处理。

`url_kind` 一并留在表里，下载环节应当按它分流，而不是无差别地 GET。

In [ ]:
PRIORITY = [
    lambda c: c["via"].startswith("badge:") and c["url_kind"] == "pdf",
    lambda c: c.get("direct_pdf_flag") and c["url_kind"] == "pdf",
    lambda c: c["via"].startswith("badge:"),
    lambda c: True,
]


def pick_pdf_url(candidates: list[dict]) -> dict:
    for tier, rule in enumerate(PRIORITY, 1):
        for c in candidates:
            if rule(c):
                return {"pdf_url": c["url"], "pdf_via": c["via"],
                        "url_kind": c["url_kind"], "url_host": c["host"], "pick_tier": tier}
    return {"pdf_url": None, "pdf_via": None, "url_kind": None, "url_host": None, "pick_tier": None}


df = pd.DataFrame([r | pick_pdf_url(r["candidates"]) for r in records])
df["n_candidates"] = df["candidates"].str.len()
cols = ["query", "rank", "paper_id", "title", "venue", "pub_date", "citations",
        "pdf_url", "pdf_via", "url_kind", "url_host", "pick_tier", "n_candidates",
        "reader_url", "s2_url", "authors", "tldr", "source", "candidates"]
df = df[cols]
df[["query", "rank", "title", "pdf_via", "url_kind", "pdf_url"]].head(20)

## 覆盖情况

看有多少条能走到"直链"这一档，哪些只剩落地页 —— 这决定了下载环节要为多少条做人工/跳转处理。

In [ ]:
print("按选中链接形态：")
print(df["url_kind"].value_counts(dropna=False).to_string(), "\n")

print("按来源通道：")
print(df["pdf_via"].value_counts(dropna=False).to_string(), "\n")

print("按主机名：")
print(df["url_host"].value_counts(dropna=False).to_string(), "\n")

hard = df.loc[df["url_kind"] != "pdf", ["query", "title", "pdf_via", "pdf_url", "reader_url"]]
print(f"非直链（需跳转或订阅）{len(hard)} 条：")
hard

## 导出

写到 notebook 旁边的 `_scratch/`：这是探索产物，**不是** `data/processed/` 的流水线工件。
两种格式：`jsonl` 保留 `candidates` 全量线索便于排障，`csv` 给下载脚本一份扁平清单。

In [ ]:
OUT_DIR.mkdir(exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d")

jsonl_path = OUT_DIR / f"serp_pdf_urls__{stamp}.jsonl"
with jsonl_path.open("w", encoding="utf-8") as f:
    for rec in df.to_dict("records"):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

csv_path = OUT_DIR / f"serp_pdf_urls__{stamp}.csv"
flat = df.drop(columns=["candidates"]).assign(authors=df["authors"].str.join("; "))
flat.to_csv(csv_path, index=False)

print(jsonl_path.relative_to(REPO), jsonl_path.stat().st_size, "bytes")
print(csv_path.relative_to(REPO), csv_path.stat().st_size, "bytes")